[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_16/MFM_ch16_seminar/MFM_ch16_seminar.ipynb)

# MFM_ch16_seminar

Seminar 16 computations: a three-coin index and its divisor, constituent replacement; the no-arbitrage band and the optimal arbitrage trade of a constant-product pool with fees and of a weighted pool; impermanent loss of a weighted pool and its loss-versus-rebalancing rate; block-bootstrap volatility and weekend effect for Bitcoin, Ether and Solana; equilibrium threshold autoregressions for USD Coin, Tether and Dai with the Hansen (1996) fixed-regressor bootstrap; IBIT and ETHA tracking, fee drift with HAC errors, ADF and KPSS tests and Dimson betas; sup-Wald break tests for the Strategy and Coinbase betas; hedge, safe-haven, Forbes-Rigobon and quantile-regression tests for Bitcoin before and after the spot ETFs.

*Modelling Financial Markets (MFM), Chapter 16: Digital Assets and DeFi. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_16'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import json
import time
import urllib.request
import urllib.parse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from scipy.signal import lfilter
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Course colours (grey only for reference lines and the grid)
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Teal     = '#17A2B8'
Magenta  = '#D63384'
Brown    = '#795548'
Gray     = '#7F7F7F'
COL = {'BTC': Orange, 'ETH': Purple, 'XRP': MainBlue, 'ADA': Forest, 'DOGE': Amber, 'LTC': Teal, 'LINK': IDAred,
       'SOL': Magenta, 'BNB': Brown, 'SPX': MainBlue, 'GOLD': Amber, 'QQQ': Teal, 'USDT': Forest, 'USDC': MainBlue,
       'DAI': Orange, 'IBIT': MainBlue, 'ETHA': Purple, 'COIN': MainBlue, 'MSTR': IDAred, 'TLT': Forest}
CLASS_COL = {'Crypto': Orange, 'Equity': MainBlue, 'FX': Forest, 'Commodity': Amber, 'Bond': Purple}

HERE = '.'
SEED = 42
ETF_START = '2024-01-11'
B_BOOT = 2000


def save_fig(name):
    """Save the figure as a transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=4, y=0.0):
    """A single legend below a multi-panel figure."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def jsonable(x):
    """Recursively convert numpy numbers and dates to JSON types."""
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, (np.floating, float)):
        return None if not np.isfinite(x) else float(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (pd.Timestamp,)):
        return str(x.date())
    return x


def max_drawdown(p):
    """Largest fall from a previous peak (in %) and the date of the trough."""
    dd = p / p.cummax() - 1
    return 100 * dd.min(), dd.idxmin()


def hill(x, frac=0.05):
    """Hill estimator of the tail index from the largest frac of the positive values of x."""
    x = np.sort(x[x > 0])[::-1]
    k = max(int(frac * len(x)), 10)
    return 1 / np.mean(np.log(x[:k] / x[k]))


def hac_ols(y, X, lags=4):
    """OLS regression with Newey-West (HAC) standard errors."""
    return sm.OLS(y, sm.add_constant(X)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# read the course data
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# key -> (symbol, label, type, Coin Metrics id)
ASSETS = {
    'BTC': ('BTC-USD.CC', 'Bitcoin', 'crypto', 'btc'),
    'ETH': ('ETH-USD.CC', 'Ethereum', 'crypto', 'eth'),
    'XRP': ('XRP-USD.CC', 'XRP', 'crypto', 'xrp'),
    'BNB': ('BNB-USD.CC', 'BNB', 'crypto', None),
    'SOL': ('SOL-USD.CC', 'Solana', 'crypto', None),
    'ADA': ('ADA-USD.CC', 'Cardano', 'crypto', 'ada'),
    'DOGE': ('DOGE-USD.CC', 'Dogecoin', 'crypto', 'doge'),
    'LTC': ('LTC-USD.CC', 'Litecoin', 'crypto', 'ltc'),
    'LINK': ('LINK-USD.CC', 'Chainlink', 'crypto', 'link'),
    'USDT': ('USDT-USD.CC', 'Tether (USDT)', 'crypto', None),
    'USDC': ('USDC-USD.CC', 'USD Coin (USDC)', 'crypto', None),
    'DAI': ('DAI-USD.CC', 'Dai', 'crypto', None),
    'IBIT': ('IBIT.US', 'IBIT (spot Bitcoin ETF)', 'etf', None),
    'ETHA': ('ETHA.US', 'ETHA (spot Ether ETF)', 'etf', None),
    'COIN': ('COIN.US', 'Coinbase', 'etf', None),
    'MSTR': ('MSTR.US', 'Strategy', 'etf', None),
    'QQQ': ('QQQ.US', 'Nasdaq 100 (QQQ)', 'etf', None),
    'GLD': ('GLD.US', 'Gold (GLD)', 'etf', None),
    'TLT': ('TLT.US', 'Long Treasuries (TLT)', 'etf', None),
    'SPX': ('GSPC.INDX', 'S&P 500', 'index', None),
    'GOLD': ('XAUUSD.FOREX', 'Gold (XAU/USD)', 'fx', None),
}
LABELS = {k: v[1] for k, v in ASSETS.items()}
# assets with a price series whose public current supply equals the circulating supply
# (XRP and Chainlink: the reported supply includes tokens held by the issuer; Solana, BNB: no public supply)
CRIX_UNIVERSE = ['BTC', 'ETH', 'DOGE', 'ADA', 'LTC']
STABLE = ['USDT', 'USDC', 'DAI']

_CACHE = {}
UA = {'User-Agent': 'Mozilla/5.0 (MFM course notebook)'}


def _get_json(url, tries=4):
    """Read a JSON response."""
    for i in range(tries):
        try:
            return json.load(urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=120))
        except Exception:
            if i == tries - 1:
                raise
            time.sleep(3 * (i + 1))


def read_market(symbol):
    """Read the daily price series of one asset from the course data."""
    if symbol in _CACHE:
        return _CACHE[symbol]
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    _CACHE[symbol] = pd.read_csv(src, index_col='date', parse_dates=True).sort_index()
    return _CACHE[symbol]


def price(key, start=None, end=END, col=None):
    """Cleaned daily price: crypto 7/7 (close), ETFs/shares (adjusted close), indices/FX (close) on weekdays."""
    symbol, _, kind, _ = ASSETS[key]
    d = read_market(symbol)
    col = col or ('adjusted_close' if kind == 'etf' else 'close')
    s = d[col].loc[start:end]
    s = s[s > 0].dropna()
    if kind != 'crypto':
        s = s[s.index.dayofweek < 5]
    if kind == 'index':
        s = s[s.diff() != 0]
    return s.rename(key)


def log_returns(key, start=None, end=END):
    """Log returns on the series' own calendar."""
    return np.log(price(key, start, end)).diff().dropna().rename(key)


def joint_prices(keys, start=None, end=END):
    """Prices of several assets on COMMON days (prices aligned first)."""
    return pd.concat([price(k, None, end) for k in keys], axis=1, join='inner').dropna().loc[start:]


def joint_returns(keys, start=None, end=END, freq=None):
    """Common log returns: align prices first, optional weekly (Friday) sampling, then returns."""
    p = joint_prices(keys, None, end)
    if freq == 'W':
        p = p.resample('W-FRI').last().dropna()
    return np.log(p).diff().dropna().loc[start:]


def periods_per_year(r):
    """Actual frequency: average number of observations per calendar year."""
    return len(r) / ((r.index[-1] - r.index[0]).days / 365.25)


def coin_supply(asset, start='2017-01-01', end=END):
    """Daily current supply (SplyCur) of a crypto-asset, from Coin Metrics Community Data."""
    key = ('supply', asset, start, end)
    if key in _CACHE:
        return _CACHE[key]
    url = ('https://community-api.coinmetrics.io/v4/timeseries/asset-metrics?assets=' + asset +
           f'&metrics=SplyCur&frequency=1d&start_time={start}&end_time={end}&page_size=10000')
    rows = []
    while url:
        js = _get_json(url)
        rows += js.get('data', [])
        url = js.get('next_page_url')
    s = pd.Series({pd.Timestamp(r['time'][:10]): float(r['SplyCur']) for r in rows}).sort_index()
    _CACHE[key] = s.rename(asset)
    return _CACHE[key]


def market_values(keys=CRIX_UNIVERSE, start='2018-01-01', end=END):
    """Daily market value = closing price x current supply, in billion USD."""
    out = {}
    for k in keys:
        p = price(k, start, end)
        s = coin_supply(ASSETS[k][3], start, end).reindex(p.index).ffill()
        out[k] = p * s / 1e9
    return pd.DataFrame(out).dropna()


def defi_tvl():
    """Total value locked (TVL) in DeFi protocols, all blockchains, billion USD (DefiLlama)."""
    if 'tvl' not in _CACHE:
        js = _get_json('https://api.llama.fi/v2/historicalChainTvl')
        s = pd.Series({pd.Timestamp(int(x['date']), unit='s').normalize(): x['tvl'] / 1e9 for x in js}).sort_index()
        _CACHE['tvl'] = s[s > 0].loc[:END].rename('TVL')
    return _CACHE['tvl']


def stablecoin_chart(sid=None):
    """Circulating supply (billion units) and its value (billion USD); sid=None: all USD stablecoins."""
    key = ('stable', sid)
    if key not in _CACHE:
        url = 'https://stablecoins.llama.fi/stablecoincharts/all' + (f'?stablecoin={sid}' if sid else '')
        js = _get_json(url)
        rows = {}
        for x in js:
            d = pd.Timestamp(int(x['date']), unit='s').normalize()
            c = (x.get('totalCirculating') or {}).get('peggedUSD')
            v = (x.get('totalCirculatingUSD') or {}).get('peggedUSD')
            if c is not None:
                rows[d] = (c / 1e9, (v if v is not None else np.nan) / 1e9)
        df = pd.DataFrame(rows, index=['supply', 'value']).T.sort_index()
        _CACHE[key] = df.loc[:END]
    return _CACHE[key]


def stablecoin_list():
    """Today's USD stablecoins: symbol, name, peg mechanism, supply (billion USD)."""
    if 'slist' not in _CACHE:
        js = _get_json('https://stablecoins.llama.fi/stablecoins?includePrices=false')['peggedAssets']
        rows = [dict(id=x['id'], symbol=x['symbol'], name=x['name'], mechanism=x.get('pegMechanism'),
                     supply=((x.get('circulating') or {}).get('peggedUSD') or 0) / 1e9)
                for x in js if x.get('pegType') == 'peggedUSD']
        _CACHE['slist'] = pd.DataFrame(rows).sort_values('supply', ascending=False).reset_index(drop=True)
    return _CACHE['slist']


def chain_tvl():
    """Today's TVL by blockchain, billion USD (DefiLlama)."""
    if 'chains' not in _CACHE:
        js = _get_json('https://api.llama.fi/v2/chains')
        s = pd.Series({x['name']: (x.get('tvl') or 0) / 1e9 for x in js}).sort_values(ascending=False)
        _CACHE['chains'] = s
    return _CACHE['chains']


def chain_tvl_at(name, date=END):
    """TVL of one blockchain on a date (last value up to that date), billion USD (DefiLlama)."""
    key = ('chain', name)
    if key not in _CACHE:
        js = _get_json('https://api.llama.fi/v2/historicalChainTvl/' + urllib.parse.quote(name))
        _CACHE[key] = pd.Series({pd.Timestamp(int(x['date']), unit='s').normalize(): x['tvl'] / 1e9 for x in js}).sort_index()
    s = _CACHE[key].loc[:date]
    return float(s.iloc[-1]) if len(s) else 0.0


# universe for the asset-class map (a reduced replication of Pele et al., 2023): symbol -> (label, class, type)
CLASS_ASSETS = {
    'BTC-USD.CC': ('Bitcoin', 'Crypto', 'crypto'), 'ETH-USD.CC': ('Ethereum', 'Crypto', 'crypto'),
    'XRP-USD.CC': ('XRP', 'Crypto', 'crypto'), 'LTC-USD.CC': ('Litecoin', 'Crypto', 'crypto'),
    'DOGE-USD.CC': ('Dogecoin', 'Crypto', 'crypto'), 'ADA-USD.CC': ('Cardano', 'Crypto', 'crypto'),
    'LINK-USD.CC': ('Chainlink', 'Crypto', 'crypto'), 'BNB-USD.CC': ('BNB', 'Crypto', 'crypto'),
    'GSPC.INDX': ('S&P 500', 'Equity', 'index'), 'NDX.INDX': ('Nasdaq 100', 'Equity', 'index'),
    'STOXX50E.INDX': ('Euro Stoxx 50', 'Equity', 'index'), 'GDAXI.INDX': ('DAX', 'Equity', 'index'),
    'N225.INDX': ('Nikkei 225', 'Equity', 'index'), 'BET': ('BET', 'Equity', 'index'),
    'EURUSD.FOREX': ('EUR/USD', 'FX', 'fx'), 'USDJPY.FOREX': ('USD/JPY', 'FX', 'fx'),
    'GBPUSD.FOREX': ('GBP/USD', 'FX', 'fx'), 'USDCHF.FOREX': ('USD/CHF', 'FX', 'fx'),
    'XAUUSD.FOREX': ('Gold', 'Commodity', 'fx'), 'USO.US': ('Oil (USO)', 'Commodity', 'etf'),
    'UNG.US': ('Natural gas (UNG)', 'Commodity', 'etf'), 'DBC.US': ('Commodities (DBC)', 'Commodity', 'etf'),
    'TLT.US': ('Long Treasuries (TLT)', 'Bond', 'etf'), 'IEF.US': ('7-10y Treasuries (IEF)', 'Bond', 'etf'),
    'LQD.US': ('Corporate IG (LQD)', 'Bond', 'etf'), 'HYG.US': ('High yield (HYG)', 'Bond', 'etf'),
}


def symbol_returns(symbol, start=None, end=END):
    """Daily log returns of a symbol from CLASS_ASSETS, on its own calendar."""
    _, _, kind = CLASS_ASSETS[symbol]
    d = read_market(symbol)
    s = d['adjusted_close' if kind == 'etf' else 'close'].loc[:end]
    s = s[s > 0].dropna()
    if kind != 'crypto':
        s = s[s.index.dayofweek < 5]
    if kind == 'index':
        s = s[s.diff() != 0]
    return np.log(s).diff().dropna().loc[start:end]

## Helper functions

In [ ]:
BLOCK = 20

TRIM = 0.15

def amm_swap(x, y, dx, fee=0.003):
    """Swap dx of asset X in a pool x*y = k with a fee: amount received and effective price."""
    g = 1 - fee
    dy = y * g * dx / (x + g * dx)
    return dy, dy / dx


def impermanent_loss(r):
    """Impermanent loss of a 50/50 liquidity provider in x*y = k, for the price ratio r = P1/P0."""
    return 2 * np.sqrt(r) / (1 + r) - 1


def etf_tracking(etf='IBIT', coin='BTC'):
    """ETF vs underlying returns on the ETF's trading days: daily (same date and next date) and weekly (Friday);
    tracking error; drift of the ETF / underlying ratio (the annual fee)."""
    p = joint_prices([etf, coin])
    r = np.log(p).diff().dropna()
    c_next = np.log(price(coin)).diff().shift(-1).reindex(r.index)
    d_same = hac_ols(r[etf], r[coin])
    xx = pd.concat([r[coin], c_next.rename('next')], axis=1).dropna()
    d_two = hac_ols(r[etf].reindex(xx.index), xx)
    w = joint_returns([etf, coin], freq='W')
    wk = hac_ols(w[etf], w[coin])
    te_d = 100 * (r[etf] - r[coin]).std() * np.sqrt(252)
    te_w = 100 * (w[etf] - w[coin]).std() * np.sqrt(52)
    lr = np.log(p[etf] / p[coin])
    t = (lr.index - lr.index[0]).days / 365.25
    drift = np.polyfit(t, lr.values, 1)[0]
    vol = read_market(ASSETS[etf][0])
    dv = (vol['close'] * vol['volume']).loc[:END] / 1e9
    return dict(start=str(p.index[0].date()), n_d=len(r), n_w=len(w), beta_d=d_same.params[coin], r2_d=d_same.rsquared,
                beta_d_next=d_two.params['next'], beta_d_same=d_two.params[coin], r2_d_two=d_two.rsquared,
                beta_w=wk.params[coin], se_w=wk.bse[coin], r2_w=wk.rsquared, te_d=te_d, te_w=te_w,
                drift=100 * drift, corr_next=r[etf].corr(c_next), dv_mean=dv.mean(), dv_last60=dv.iloc[-60:].mean(),
                dv_max=dv.max(), dv_max_date=str(dv.idxmax().date()))


def nw_lags(T):
    """Number of Newey-West lags: floor(4 (T/100)^(2/9))."""
    return int(np.floor(4 * (T / 100) ** (2 / 9)))


def hac_meat(Z, u, L):
    """Long-run variance matrix of z_t u_t, Bartlett kernel with L lags."""
    g = Z * u[:, None]
    S = g.T @ g
    for l in range(1, L + 1):
        G = g[l:].T @ g[:-l]
        S += (1 - l / (L + 1)) * (G + G.T)
    return S


def sup_wald_crit(p, trim=TRIM, reps=20000, grid=1000, seed=SEED):
    """Asymptotic null distribution of sup-Wald (Andrews, 1993): sup ||B(r) - r B(1)||^2 / (r (1 - r)),
    B a p-dimensional Brownian motion, r in [trim, 1 - trim]; simulation."""
    rng = np.random.default_rng(seed)
    r = np.arange(1, grid + 1) / grid
    sel = (r >= trim) & (r <= 1 - trim)
    out = np.empty(reps)
    for i in range(reps):
        W = np.cumsum(rng.standard_normal((grid, p)), axis=0) / np.sqrt(grid)
        BB = W - r[:, None] * W[-1]
        out[i] = np.max((BB[sel] ** 2).sum(axis=1) / (r[sel] * (1 - r[sel])))
    return out


def sup_wald(y, X, index, trim=TRIM, lags=None):
    """sup-Wald test for a break in all coefficients of y = X b + u at an unknown date, with a HAC (Bartlett)
    covariance matrix computed separately in each regime; estimated date = minimum SSR (Bai, 1997)."""
    y, X = np.asarray(y, float), np.asarray(X, float)
    T, p = X.shape
    L = nw_lags(T) if lags is None else lags
    lo, hi = int(np.floor(trim * T)), int(np.ceil((1 - trim) * T))
    W, SSR = np.full(T, np.nan), np.full(T, np.nan)
    for tb in range(lo, hi):
        res = []
        V = np.zeros((p, p))
        ssr = 0.0
        for a, b in [(0, tb), (tb, T)]:
            Xa, ya = X[a:b], y[a:b]
            XtX = Xa.T @ Xa
            bet = np.linalg.solve(XtX, Xa.T @ ya)
            u = ya - Xa @ bet
            ssr += u @ u
            Q = np.linalg.inv(XtX)
            V += Q @ hac_meat(Xa, u, L) @ Q
            res.append(bet)
        d = res[1] - res[0]
        W[tb] = d @ np.linalg.solve(V, d)
        SSR[tb] = ssr
    tb_w = int(np.nanargmax(W))
    tb_hat = int(np.nanargmin(SSR))
    # Bai (1997) CI: (d'Q d)^2 / (d'Omega d) (T_hat - T0) -> argmax(W(s) - |s|/2); 97.5% quantile = 11
    b0 = np.linalg.lstsq(X[:tb_hat], y[:tb_hat], rcond=None)[0]
    b1 = np.linalg.lstsq(X[tb_hat:], y[tb_hat:], rcond=None)[0]
    d = b1 - b0
    u = y - np.where(np.arange(T)[:, None] < tb_hat, X @ b0[:, None], X @ b1[:, None]).ravel()
    Qm = X.T @ X / T
    Om = hac_meat(X, u, L) / T
    Lm = (d @ Qm @ d) ** 2 / (d @ Om @ d)
    half = int(np.ceil(11 / Lm)) + 1
    idx = pd.DatetimeIndex(index)
    return dict(sup_w=float(np.nanmax(W)), date_w=str(idx[tb_w].date()), date=str(idx[tb_hat].date()),
                ci_lo=str(idx[max(tb_hat - half, 0)].date()), ci_hi=str(idx[min(tb_hat + half, T - 1)].date()),
                half=half, b_pre=b0.tolist(), b_post=b1.tolist(), T=T, lags=L, W=W)


def tar_fit(d, trim=TRIM, B=999, seed=SEED):
    """Equilibrium threshold AR (EQ-TAR, Balke & Fomby 1997): d_t = phi_in d_{t-1} + e_t if |d_{t-1}| <= c,
    d_t = phi_out d_{t-1} + e_t otherwise.
    Threshold c: grid search (values of |d_{t-1}| between the trim and 1 - trim quantiles), minimum SSR.
    Test H0: phi_in = phi_out (linear AR(1)), heteroskedasticity-robust sup-Wald; p-value by the fixed-regressor
    bootstrap y*_t = e_t eta_t, eta_t ~ N(0, 1), e_t the residuals of the linear model (H0) (Hansen, 1996).
    Only pairs (d_(t-1), d_t) on consecutive calendar days: if a period is removed from the series, no
    pair crosses the gap."""
    ok = np.diff(d.index.values).astype('timedelta64[D]') == np.timedelta64(1, 'D')
    y, x = d.values[1:][ok], d.values[:-1][ok]
    a = np.abs(x)
    order = np.argsort(a, kind='stable')
    xs, ys, as_ = x[order], y[order], a[order]
    n = len(y)
    x2 = xs ** 2
    lo, hi = int(np.floor(trim * n)), int(np.ceil((1 - trim) * n))
    # positions where the sample can be split (distinct values of |x|)
    cuts = np.array([i for i in range(lo, hi) if as_[i] < as_[i + 1]]) + 1   # the inner regime = the first i observations

    def wald(yy):
        c1 = np.cumsum(xs * yy)
        c2 = np.cumsum(x2)
        c3 = np.cumsum(x2 * yy ** 2)
        c4 = np.cumsum(xs ** 3 * yy)
        c5 = np.cumsum(xs ** 4)
        T1, T2, T3, T4, T5 = c1[-1], c2[-1], c3[-1], c4[-1], c5[-1]
        i = cuts - 1
        Sin, Sout = c2[i], T2 - c2[i]
        pin, pout = c1[i] / Sin, (T1 - c1[i]) / Sout
        vin = (c3[i] - 2 * pin * c4[i] + pin ** 2 * c5[i]) / Sin ** 2
        vout = ((T3 - c3[i]) - 2 * pout * (T4 - c4[i]) + pout ** 2 * (T5 - c5[i])) / Sout ** 2
        return (pin - pout) ** 2 / (vin + vout), pin, pout, vin, vout

    Wc, pin, pout, vin, vout = wald(ys)
    # threshold estimate: minimum SSR
    c1 = np.cumsum(xs * ys)
    c2 = np.cumsum(x2)
    i = cuts - 1
    ssr = np.sum(ys ** 2) - c1[i] ** 2 / c2[i] - (c1[-1] - c1[i]) ** 2 / (c2[-1] - c2[i])
    j = int(np.argmin(ssr))
    phi_lin = np.sum(x * y) / np.sum(x ** 2)
    e0 = ys - phi_lin * xs
    rng = np.random.default_rng(seed)
    sup0 = Wc.max()
    bs = np.array([wald(e0 * rng.standard_normal(n))[0].max() for _ in range(B)])
    hl = lambda p: float(np.log(0.5) / np.log(abs(p))) if 0 < abs(p) < 1 else None
    nin = int(cuts[j])
    return dict(n=n, c=float(as_[cuts[j] - 1]), n_in=nin, n_out=n - nin, share_out=100 * (n - nin) / n,
                phi_in=float(pin[j]), se_in=float(np.sqrt(vin[j])), phi_out=float(pout[j]), se_out=float(np.sqrt(vout[j])),
                hl_in=hl(pin[j]), hl_out=hl(pout[j]), phi_lin=float(phi_lin), hl_lin=hl(phi_lin),
                sup_w=float(sup0), p_boot=float((1 + (bs >= sup0).sum()) / (B + 1)), B=B,
                chi2_p=float(stats.chi2.sf(sup0, 1)), c_grid=[float(as_[cuts[0] - 1]), float(as_[cuts[-1] - 1])],
                share_var_out=float(100 * np.sum(x2[nin:]) / np.sum(x2)))


def dimson(etf='IBIT', coin='BTC'):
    """r_ETF,t = a + b_-1 r_coin,t-1 + b_0 r_coin,t + b_+1 r_coin,t+1 + u_t, on common days; Dimson beta = the sum,
    Newey-West standard error."""
    r = 100 * joint_returns([etf, coin])
    X = pd.concat([r[coin].shift(1).rename('lag'), r[coin].rename('now'), r[coin].shift(-1).rename('lead')], axis=1)
    d = pd.concat([r[etf], X], axis=1).dropna()
    m = sm.OLS(d[etf], sm.add_constant(d[['lag', 'now', 'lead']])).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    R = np.array([0, 1, 1, 1])
    s = float(R @ m.params.values)
    se = float(np.sqrt(R @ m.cov_params().values @ R))
    m0 = sm.OLS(d[etf], sm.add_constant(d['now'])).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    return dict(n=len(d), b_lag=float(m.params['lag']), b_now=float(m.params['now']), b_lead=float(m.params['lead']),
                se_lead=float(m.bse['lead']), sum=s, se=se, t1=(s - 1) / se, b_ols=float(m0.params['now']),
                se_ols=float(m0.bse['now']), r2=float(m.rsquared))

## Analysis

In [ ]:
def a1_index():
    """Laspeyres index with three coins: level, divisor, reweighting at the end of the month."""
    p0 = np.array([60000.0, 3000.0, 150.0])          # prices on the base date
    q0 = np.array([19.5e6, 120e6, 500e6])            # circulating supply on the base date
    mv0 = p0 * q0
    D0 = mv0.sum() / 1000                            # divisor: the index starts at 1000
    p1 = np.array([66000.0, 2700.0, 180.0])          # prices at the end of the month
    I1 = (p1 * q0).sum() / D0
    w0 = mv0 / mv0.sum()
    q1 = np.array([19.52e6, 120.1e6, 560e6])         # new supply at reweighting
    D1 = (p1 * q1).sum() / I1                        # new divisor: the same level before and after
    return dict(mv0=mv0.tolist(), D0=D0, w0=w0.tolist(), I1=I1, ret=I1 / 1000 - 1, D1=D1,
                w1=((p1 * q1) / (p1 * q1).sum()).tolist(), rets=(p1 / p0 - 1).tolist())


def a2_replace():
    """Replacing a constituent: coin C leaves, coin D enters; the new divisor keeps the level."""
    p = np.array([66000.0, 2700.0, 180.0])
    q = np.array([19.52e6, 120.1e6, 560e6])
    I = 1000 * 1.0
    base = a1_index()
    I = base['I1']
    D = base['D1']
    pD, qD = 0.5, 40e9
    new_mv = p[0] * q[0] + p[1] * q[1] + pD * qD
    D_new = new_mv / I
    return dict(old_mv=(p * q).sum(), new_mv=new_mv, D_old=D, D_new=D_new, I=I, mvD=pD * qD, mvC=p[2] * q[2])


def a3_band(x=1000.0, y=3_000_000.0, P=3300.0, fee=0.003):
    """No-arbitrage band of a pool x*y = k with fee f: gamma P <= p <= P / gamma; optimal size of the
    arbitrage when the outside price P leaves the band: effective reserve y' = y + gamma dy* = sqrt(gamma P k),
    dy* = (y' - y) / gamma. The fee stays in the pool (Uniswap v2 convention): the actual reserves after the trade are
    x' and y + dy*, so the pool price is (y + dy*) / x'. Profit per unit of pool value: relative to the
    pool value before the trade, at the pool price (p x + y)."""
    g = 1 - fee
    k = x * y
    p = y / x
    y1 = np.sqrt(g * P * k)
    x1 = k / y1
    dy = (y1 - y) / g
    dx = x - x1
    profit = dx * P - dy
    V0 = p * x + y
    y_act = y + dy
    return dict(k=k, p=p, lo=g * P, hi=P / g, y1=y1, x1=x1, dy=dy, dx=dx, profit=profit, p_eff=y1 / x1,
                y_act=y_act, p_new=y_act / x1, band_lo=g * y_act / x1, band_hi=y_act / x1 / g,
                avg=dy / dx, profit_nofee=a4_nofee(x, y, P), V0=V0, rel=100 * profit / V0)


def a4_nofee(x, y, P):
    """Arbitrage profit without a fee (same pool): P (x - x1) - (y1 - y), x1 = sqrt(k/P), y1 = sqrt(kP)."""
    k = x * y
    return P * (x - np.sqrt(k / P)) - (np.sqrt(k * P) - y)


def a4_weighted(w=0.8, x=1000.0, p=3000.0, P=3300.0, fee=0.003):
    """Weighted pool x^w y^(1-w) = k (Balancer type): marginal price p = (w/(1-w)) y/x; effective reserves after
    arbitrage x' = k (w/(gamma P (1-w)))^(1-w), y' = k (gamma P (1-w)/w)^w; dy* = (y' - y)/gamma. The fee stays in the
    pool: actual reserve y + dy*, pool price (w/(1-w)) (y + dy*)/x'. Profit per unit of value: relative to the
    pool value before the trade, p x + y = y / (1 - w)."""
    g = 1 - fee
    y = p * x * (1 - w) / w
    k = x ** w * y ** (1 - w)
    x1 = k * (w / (g * P * (1 - w))) ** (1 - w)
    y1 = k * (g * P * (1 - w) / w) ** w
    dy = (y1 - y) / g
    dx = x - x1
    V0 = p * x + y
    y_act = y + dy
    return dict(w=w, y=y, k=k, x1=x1, y1=y1, dy=dy, dx=dx, profit=dx * P - dy, p_eff=(w / (1 - w)) * y1 / x1,
                y_act=y_act, p_new=(w / (1 - w)) * y_act / x1, V0=V0, rel=100 * (dx * P - dy) / V0)


def il_weighted(r, w):
    """Impermanent loss of a weighted pool: r^w / (w r + 1 - w) - 1."""
    return r ** w / (w * r + 1 - w) - 1


def a5_il():
    """IL for w = 0.5 and w = 0.8 at r = 0.5, 2, 4; second-order approximation -w(1-w)(ln r)^2 / 2."""
    out = {}
    for w in [0.5, 0.8]:
        for r in [0.5, 2.0, 4.0]:
            out[f'w{w:g}_r{r:g}'] = 100 * il_weighted(r, w)
            out[f'q{w:g}_r{r:g}'] = -100 * w * (1 - w) * np.log(r) ** 2 / 2
    return out


def a6_lvr(sig=0.70):
    """LVR rate of a weighted pool: sigma^2 w (1 - w) / 2; w = 1/2 gives sigma^2 / 8."""
    return {f'w{w:g}': 100 * sig ** 2 * w * (1 - w) / 2 for w in [0.5, 0.8, 0.95]}


def a7_tar(tar):
    """Algebra of the equilibrium threshold AR (EQ-TAR, Balke & Fomby 1997) on the USDC estimates of B3: half-lives
    in each regime and the number of days until a deviation of -285 bp (the close of 11 March 2023) re-enters the band."""
    d0 = 285.0
    n_out = np.log(tar['c'] / d0) / np.log(tar['phi_out'])
    return dict(d0=d0, n_out=n_out, n_days=int(np.ceil(n_out)), hl_in=tar['hl_in'], hl_out=tar['hl_out'])


def a8_mix(tar):
    """Probability limit of linear OLS on EQ-TAR data: phi_lin = phi_in (1 - s) + phi_out s,
    s = share of sum d_{t-1}^2 outside the band."""
    s = tar['share_var_out'] / 100
    return dict(s=100 * s, mix=tar['phi_in'] * (1 - s) + tar['phi_out'] * s, phi_lin=tar['phi_lin'])


def block_boot(x, stat, B=B_BOOT, block=BLOCK, seed=SEED):
    """Moving-block bootstrap (Kunsch, 1989) for a statistic of a DataFrame/Series; 95% percentile interval."""
    rng = np.random.default_rng(seed)
    n = len(x)
    nb = int(np.ceil(n / block))
    vals = []
    for _ in range(B):
        st = rng.integers(0, n - block + 1, nb)
        idx = np.concatenate([np.arange(s, s + block) for s in st])[:n]
        vals.append(stat(x.iloc[idx]))
    vals = np.array(vals)
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5), vals


def weekend_ratio(r):
    """Ratio of (centred) variances: weekend (Saturday, Sunday) / weekdays."""
    we = r[r.index.dayofweek >= 5]
    wd = r[r.index.dayofweek < 5]
    return we.var(ddof=1) / wd.var(ddof=1)


def b1_btc_facts(key='BTC'):
    """Annualised volatility relative to the S&P 500 and the weekend effect before / after the ETFs, with a block bootstrap."""
    r = 100 * log_returns(key, '2018-01-01')
    s = 100 * log_returns('SPX', str(r.index[0].date()))
    vc = r.std() * np.sqrt(365)
    vs = s.std() * np.sqrt(252)
    lo, hi, _ = block_boot(r, lambda x: x.std() * np.sqrt(365))
    pre, post = r.loc[:'2024-01-10'], r.loc[ETF_START:]
    rp, rq = weekend_ratio(pre), weekend_ratio(post)
    plo, phi_, dpre = block_boot(pre, weekend_ratio, block=21)
    qlo, qhi, dpost = block_boot(post, weekend_ratio, block=21, seed=SEED + 1)
    diff = dpost - dpre
    return dict(start=str(r.index[0].date()), n=len(r), vol=vc, vol_lo=lo, vol_hi=hi, vol_spx=vs, ratio=vc / vs,
                wr_pre=rp, wr_pre_lo=plo, wr_pre_hi=phi_, wr_post=rq, wr_post_lo=qlo, wr_post_hi=qhi,
                diff=rq - rp, diff_lo=np.percentile(diff, 2.5), diff_hi=np.percentile(diff, 97.5),
                n_pre=len(pre), n_post=len(post), kurt=float(pd.Series(r).kurt()), _dpre=dpre, _dpost=dpost)


def fig_sem_weekend(b1):
    """Bootstrap distributions of the weekend / weekday ratio, before and after the ETFs."""
    fig, ax = plt.subplots(figsize=(6.8, 2.8))
    ax.hist(b1['_dpre'], bins=50, color=MainBlue, alpha=0.75, density=True, label='2018 - 10 Jan 2024')
    ax.hist(b1['_dpost'], bins=50, color=Orange, alpha=0.75, density=True, label='11 Jan 2024 - Sep 2026')
    ax.axvline(1, color=Gray, lw=0.7, ls='--')
    ax.set_xlabel('Weekend variance / weekday variance (Bitcoin), block bootstrap draws')
    ax.set_ylabel('Density')
    legend_outside_bottom(ax, ncol=2, y=-0.2)
    save_fig('ch16_sem_weekend')


def b3_usdc(key='USDC', start='2021-01-01'):
    """Daily peg deviation (bp): statistics, equilibrium threshold AR (EQ-TAR) with the sup-Wald test and a
    fixed-regressor bootstrap (Hansen, 1996), with and without the week of 9-16 March 2023."""
    d = read_market(ASSETS[key][0]).loc[start:END]
    dev = 1e4 * (d['close'] - 1)
    ex = dev.drop(dev.loc['2023-03-09':'2023-03-16'].index)
    return dict(key=key, n=len(dev), sd=dev.std(), sd_ex=ex.std(), out50=int((dev.abs() > 50).sum()),
                out50_ex=int((ex.abs() > 50).sum()), min=dev.min(), min_date=str(dev.idxmin().date()),
                low=1e4 * (d['low'].min() - 1), low_date=str(d['low'].idxmin().date()),
                full=I.tar_fit(dev), ex=I.tar_fit(ex), kurt=float(dev.kurt()))


def fig_sem_usdc(tar):
    """USDC: peg deviation in 2023 and today's deviation against yesterday's, with the EQ-TAR lines of each
    regime (estimated threshold c) and the linear AR(1) line."""
    d = read_market(ASSETS['USDC'][0]).loc['2021-01-01':END]
    dev = 1e4 * (d['close'] - 1)
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9))
    z = dev.loc['2023-01-01':'2023-06-30']
    axes[0].plot(z.index, z.values, color=MainBlue, lw=0.9, label='USDC close deviation (bp)')
    axes[0].axhline(0, color=Gray, lw=0.5)
    axes[0].tick_params(axis='x', labelsize=7, rotation=30)
    axes[0].set_ylabel('bp')
    z1 = pd.concat([dev.shift(1), dev], axis=1).dropna()
    z1.columns = ['lag', 'now']
    zz = z1[(z1.abs() <= 12).all(axis=1)]
    axes[1].scatter(zz['lag'], zz['now'], s=5, color=Teal, alpha=0.6, label='Pairs of days (within 12 bp)')
    c = tar['c']
    xi = np.array([-c, c])
    axes[1].plot(xi, tar['phi_in'] * xi, color=Forest, lw=1.8, label=f'Inside the band: slope {tar["phi_in"]:.2f}')
    for xo in [np.array([-12, -c]), np.array([c, 12])]:
        axes[1].plot(xo, tar['phi_out'] * xo, color=IDAred, lw=1.8)
    axes[1].plot([], [], color=IDAred, lw=1.8, label=f'Outside the band: slope {tar["phi_out"]:.2f}')
    for v in [-c, c]:
        axes[1].axvline(v, color=Gray, lw=0.6, ls='--')
    axes[1].set_xlim(-12, 12)
    axes[1].set_ylim(-12, 12)
    axes[1].set_xlabel('Deviation yesterday (bp)')
    axes[1].set_ylabel('Deviation today (bp)')
    fig.tight_layout()
    fig_legend_bottom(fig, ncol=2, y=0.0)
    save_fig('ch16_sem_usdc')


def b5_etf(etf='IBIT', coin='BTC'):
    """Tracking: daily vs weekly beta (HAC); the fee drift as the mean of Delta ln(P_ETF/P_coin) with a HAC
    error; ADF (H0: unit root) and KPSS (H0: stationarity around a trend) tests on the level of the ratio;
    comparison with the slope of the level on time; Dimson (1979) beta with one lag and one lead."""
    from statsmodels.tsa.stattools import adfuller, kpss
    t = etf_tracking(etf, coin)
    p = pd.concat([price(etf), price(coin)], axis=1, join='inner').dropna()
    lr = np.log(p[etf] / p[coin])
    yrs = (lr.index[-1] - lr.index[0]).days / 365.25
    dl = lr.diff().dropna()
    ppy = len(dl) / yrs
    L = I.nw_lags(len(dl))
    m0 = sm.OLS(dl.values, np.ones(len(dl))).fit(cov_type='HAC', cov_kwds={'maxlags': L})
    tt = pd.Series((lr.index - lr.index[0]).days / 365.25, index=lr.index)
    m = hac_ols(lr, tt, lags=20)
    adf = adfuller(lr.values, regression='ct', autolag='AIC')
    kp = kpss(lr.values, regression='ct', nlags='auto')
    adf_d = adfuller(dl.values, regression='c', autolag='AIC')
    t.update(drift_mean=100 * ppy * m0.params[0], drift_mse=100 * ppy * m0.bse[0],
             drift_mlo=100 * ppy * (m0.params[0] - 1.96 * m0.bse[0]), drift_mhi=100 * ppy * (m0.params[0] + 1.96 * m0.bse[0]),
             drift_hac=100 * m.params.iloc[1], drift_se=100 * m.bse.iloc[1],
             drift_lo=100 * (m.params.iloc[1] - 1.96 * m.bse.iloc[1]), drift_hi=100 * (m.params.iloc[1] + 1.96 * m.bse.iloc[1]),
             adf=adf[0], adf_p=adf[1], kpss=kp[0], kpss_p=kp[1], adf_d=adf_d[0], adf_d_p=adf_d[1],
             t_beta_w=(t['beta_w'] - 1) / t['se_w'], lags=L, dimson=I.dimson(etf, coin))
    return t


def b7_equity(key='MSTR'):
    """Weekly two-factor regression, the test b_BTC = 1 in subperiods and the sup-Wald test (Andrews, 1993)
    for a break at an unknown date in all three coefficients, with HAC errors and the Bai (1997) CI."""
    out = {}
    # the subperiods partition the full sample: p1 ends the day before the ETFs, p2 starts with them
    for lab, a, b in [('full', '2021-04-16', END), ('p1', '2021-04-16', '2024-01-10'), ('p2', ETF_START, END)]:
        w = joint_returns([key, 'BTC', 'SPX'], None, freq='W').loc[a:b]
        m = hac_ols(w[key], w[['BTC', 'SPX']])
        out[lab] = dict(n=len(w), b_btc=m.params['BTC'], se_btc=m.bse['BTC'], b_spx=m.params['SPX'],
                        se_spx=m.bse['SPX'], r2=m.rsquared, t1=(m.params['BTC'] - 1) / m.bse['BTC'],
                        alpha=52 * 100 * m.params['const'], alpha_t=m.tvalues['const'])
    w = 100 * joint_returns([key, 'BTC', 'SPX'], None, freq='W').loc['2021-04-16':END]
    X = np.column_stack([np.ones(len(w)), w['BTC'].values, w['SPX'].values])
    sw = I.sup_wald(w[key].values, X, w.index)
    Wser = pd.Series(sw.pop('W'), index=w.index)
    crit = I.sup_wald_crit(3)
    sw.update(p=float((crit >= sw['sup_w']).mean()), cv5=float(np.percentile(crit, 95)),
              W_etf=float(Wser.loc[ETF_START:].iloc[0]), W_date=float(Wser.loc[sw['date']]))
    out['sw'] = sw
    # test of the change: full-sample regression with interactions D_t = 1 after the ETF launch, HAC errors (same lags)
    w = joint_returns([key, 'BTC', 'SPX'], None, freq='W').loc['2021-04-16':END]
    D = (w.index >= pd.Timestamp(ETF_START)).astype(float)
    Xi = pd.DataFrame({'BTC': w['BTC'], 'SPX': w['SPX'], 'D': D, 'D_BTC': D * w['BTC'], 'D_SPX': D * w['SPX']}, index=w.index)
    m = hac_ols(w[key], Xi)
    out['chg'] = dict(d_btc=m.params['D_BTC'], se=m.bse['D_BTC'], t=m.tvalues['D_BTC'], p=m.pvalues['D_BTC'])
    return out


def c1_hedge():
    """Correlations before / after the ETFs (block bootstrap) and the Baur-Lucey regression for hedge and safe haven."""
    out = {}
    per = {'pre': ('2021-01-01', '2024-01-10'), 'post': (ETF_START, END)}
    for k in ['SPX', 'QQQ', 'GLD', 'TLT']:
        r = joint_returns(['BTC', k], '2021-01-01')
        res = {}
        draws = {}
        for lab, (a, b) in per.items():
            x = r.loc[a:b]
            lo, hi, dr = block_boot(x, lambda z: z.iloc[:, 0].corr(z.iloc[:, 1]), block=20,
                                    seed=SEED + len(lab))
            res[lab] = x['BTC'].corr(x[k])
            res[lab + '_lo'], res[lab + '_hi'] = lo, hi
            draws[lab] = dr
        dd = draws['post'] - draws['pre']
        res['diff_lo'], res['diff_hi'] = np.percentile(dd, [2.5, 97.5])
        out[k] = res
    # Baur-Lucey: r_BTC = a + b r_SPX + sum_q c_q r_SPX 1{r_SPX <= q-quantile}
    bl = {}
    for lab, (a, b) in per.items():
        r = 100 * joint_returns(['BTC', 'SPX'], a).loc[a:b]
        X = pd.DataFrame({'SPX': r['SPX']})
        for q in [0.10, 0.05, 0.01]:
            thr = r['SPX'].quantile(q)
            X[f'q{int(100 * q)}'] = r['SPX'] * (r['SPX'] <= thr)
        m = hac_ols(r['BTC'], X)
        b0 = m.params['SPX']
        # the indicators are nested: b = slope above the 10% quantile, b + c10 between 5% and 10%, b + c10 + c5 between 1% and 5%,
        # b + c10 + c5 + c1 in the worst 1%; SEs of the sums from the full HAC covariance matrix
        Vc = m.cov_params()
        def tot(names):
            R = pd.Series(0.0, index=m.params.index)
            R[names] = 1.0
            return float(R @ m.params), float(np.sqrt(R @ Vc @ R))
        (t10, s10), (t5, s5), (t1, s1) = tot(['SPX', 'q10']), tot(['SPX', 'q10', 'q5']), tot(['SPX', 'q10', 'q5', 'q1'])
        mall = hac_ols(r['BTC'], r[['SPX']])          # hedge regression on all days
        bl[lab] = dict(n=len(r), b=b0, se=m.bse['SPX'], c10=m.params['q10'], c5=m.params['q5'], c1=m.params['q1'],
                       tot10=t10, tot5=t5, tot1=t1, se_tot10=s10, se_tot5=s5, se_tot1=s1,
                       lo_tot1=t1 - 1.96 * s1, hi_tot1=t1 + 1.96 * s1,
                       b_all=mall.params['SPX'], se_all=mall.bse['SPX'],
                       se10=m.bse['q10'], se5=m.bse['q5'], se1=m.bse['q1'])
        # beta on down / up days
        dn, up = r[r['SPX'] < 0], r[r['SPX'] > 0]
        bl[lab]['beta_dn'] = np.polyfit(dn['SPX'], dn['BTC'], 1)[0]
        bl[lab]['beta_up'] = np.polyfit(up['SPX'], up['BTC'], 1)[0]
        worst = r.nsmallest(10, 'SPX')
        bl[lab]['worst10_spx'] = worst['SPX'].mean()
        bl[lab]['worst10_btc'] = worst['BTC'].mean()
    out['bl'] = bl
    # Forbes-Rigobon: post-ETF correlation adjusted for the volatility of the S&P 500 (relative to the 'pre' period)
    r = joint_returns(['BTC', 'SPX'], '2021-01-01')
    x0, x1 = r.loc[per['pre'][0]:per['pre'][1]].values, r.loc[per['post'][0]:per['post'][1]].values
    def rstar(z0, z1):
        r0, r1 = np.corrcoef(z0.T)[0, 1], np.corrcoef(z1.T)[0, 1]
        dl = z1[:, 1].var() / z0[:, 1].var() - 1
        return r1 / np.sqrt(1 + dl * (1 - r1 ** 2)) - r0, dl, r1 / np.sqrt(1 + dl * (1 - r1 ** 2))
    d_fr, delta, rs = rstar(x0, x1)
    rng = np.random.default_rng(SEED)
    def mbb(z):
        n = len(z)
        st = rng.integers(0, n - 19, int(np.ceil(n / 20)))
        return z[(st[:, None] + np.arange(20)[None, :]).ravel()[:n]]
    dd = [rstar(mbb(x0), mbb(x1))[0] for _ in range(B_BOOT)]
    out['fr'] = dict(delta=delta, rho_star=rs, diff=d_fr, lo=np.percentile(dd, 2.5), hi=np.percentile(dd, 97.5))
    # quantile regression (Koenker & Bassett, 1978): quantile tau of r_BTC conditional on r_SPX, block bootstrap
    from statsmodels.regression.quantile_regression import QuantReg
    qr = {}
    for lab, (a, b) in per.items():
        z = 100 * joint_returns(['BTC', 'SPX'], a).loc[a:b]
        Xq = sm.add_constant(z['SPX'].values)
        for tau in [0.01, 0.05, 0.5]:
            bq = QuantReg(z['BTC'].values, Xq).fit(q=tau).params[1]
            bs = []
            zz = z.values
            for _ in range(300):
                y = mbb(zz)
                bs.append(QuantReg(y[:, 0], sm.add_constant(y[:, 1])).fit(q=tau).params[1])
            qr[f'{lab}_{int(100 * tau)}'] = dict(b=bq, se=float(np.std(bs)))
    out['qr'] = qr
    # volatility and the largest drawdown in the two periods
    p = price('BTC', '2021-01-01')
    for lab, (a, b) in per.items():
        x = p.loc[a:b]
        r = np.log(x).diff().dropna()
        out[lab + '_vol'] = 100 * r.std() * np.sqrt(365)
        out[lab + '_mdd'] = 100 * (x / x.cummax() - 1).min()
    return out


def fig_sem_hedge(c1):
    """Slope of Bitcoin on the S&P 500 in the four disjoint regimes of the nested-indicator regression (S&P 500 days
    above the 10% quantile, between 5% and 10%, between 1% and 5%, the worst 1%), before and after the ETFs; +- 1.96 SE."""
    fig, ax = plt.subplots(figsize=(6.8, 2.8))
    labs = ['Above 10th percentile', '5th-10th percentile', '1st-5th percentile', 'Bottom 1%']
    x = np.arange(4)
    for i, (lab, c) in enumerate([('pre', MainBlue), ('post', Orange)]):
        b = c1['bl'][lab]
        ax.bar(x + (i - 0.5) * 0.38, [b['b'], b['tot10'], b['tot5'], b['tot1']], 0.36, color=c,
               yerr=1.96 * np.array([b['se'], b['se_tot10'], b['se_tot5'], b['se_tot1']]), capsize=2,
               error_kw=dict(ecolor='black', lw=0.7),
               label='Jan 2021 - 10 Jan 2024' if lab == 'pre' else '11 Jan 2024 - Sep 2026')
    ax.axhline(0, color=Gray, lw=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(labs, fontsize=8)
    ax.set_xlabel('S&P 500 daily return regime')
    ax.set_ylabel('Bitcoin slope on the S&P 500')
    legend_outside_bottom(ax, ncol=2, y=-0.18)
    save_fig('ch16_sem_hedge')


def c2_ai(start='2021', end='2025'):
    """C2: the AI assistant's answer (log returns from prices cut to 2021-2025, annualised with 252,
    r_f = 5%) and the correction steps: 365; returns computed before selecting the period; simple excess
    returns for the Sharpe ratio; r_f = mean yield of the 3-month Treasury bill (FRED DTB3)."""
    btc = read_market(ASSETS['BTC'][0])['close']
    r = np.log(btc.loc[start:end]).diff().dropna()                  # the AI code: the first day of 2021 is lost
    out = dict(n_ai=len(r), per_year={str(k): int(v) for k, v in r.groupby(r.index.year).size().items()})
    for m in [252, 365]:
        mu, sd = r.mean() * m, r.std() * np.sqrt(m)
        out[f'mu{m}'], out[f'sd{m}'], out[f'sh{m}'] = 100 * mu, 100 * sd, (mu - 0.05) / sd
    rs = btc.pct_change().loc[start:end].dropna()                      # simple returns, computed before the selection
    tb = pd.read_csv('https://fred.stlouisfed.org/graph/fredgraph.csv?id=DTB3', index_col=0, parse_dates=True).iloc[:, 0]
    rf = float(pd.to_numeric(tb, errors='coerce').dropna().loc[start:end].mean() / 100)
    mu, sd = rs.mean() * 365, rs.std() * np.sqrt(365)
    rl = np.log1p(rs)
    out.update(n=len(rs), mu_log=100 * rl.mean() * 365, mu_s=100 * mu, sd_s=100 * sd, rf=100 * rf,
               sh_s=(mu - rf) / sd, gap=100 * (mu - rl.mean() * 365), half_var=100 * sd ** 2 / 2)
    return out


def fig_sem_a1(a1):
    """A1: market-value weights of the three coins on the base date and after the reallocation."""
    fig, ax = plt.subplots(figsize=(5.6, 2.7))
    x = np.arange(3)
    ax.bar(x - 0.19, 100 * np.array(a1['w0']), 0.36, color=MainBlue, label='Base day (weights of $D_0$)')
    ax.bar(x + 0.19, 100 * np.array(a1['w1']), 0.36, color=Orange, label='After the reallocation (weights of $D_1$)')
    for i in range(3):
        ax.text(x[i] - 0.19, 100 * a1['w0'][i] + 1.5, f"{100 * a1['w0'][i]:.1f}", ha='center', fontsize=7, color='black')
        ax.text(x[i] + 0.19, 100 * a1['w1'][i] + 1.5, f"{100 * a1['w1'][i]:.1f}", ha='center', fontsize=7, color='black')
    ax.set_xticks(x)
    ax.set_xticklabels([f"Coin {c}: return {100 * r:+.0f}%" for c, r in zip('ABC', a1['rets'])], fontsize=8)
    ax.set_ylabel('Index weight (%)')
    ax.set_ylim(0, 90)
    legend_outside_bottom(ax, ncol=2, y=-0.18)
    save_fig('ch16_sem_a1_index')


def fig_sem_a3(a3, x=1000.0, y=3_000_000.0, P=3300.0, fee=0.003):
    """A3: arbitrage profit pi(dy) = P [x - k/(y + gamma dy)] - dy, with and without a fee; the optimum is marked."""
    k = x * y
    dy = np.linspace(0, 300_000, 400)
    fig, ax = plt.subplots(figsize=(5.8, 2.8))
    for g, c, lab in [(1 - fee, MainBlue, 'Fee 0.3%'), (1.0, Orange, 'No fee')]:
        prof = P * (x - k / (y + g * dy)) - dy
        ax.plot(dy / 1000, prof / 1000, color=c, lw=1.6, label=f'{lab}: profit of buying Ether with $\\Delta y$')
    ax.scatter([a3['dy'] / 1000], [a3['profit'] / 1000], color=IDAred, s=26, zorder=3,
               label=f"Optimum with fee: $\\Delta y^*$ = {a3['dy'] / 1000:.1f}k, profit {a3['profit']:,.0f} USD")
    ax.axhline(0, color=Gray, lw=0.6)
    ax.set_xlabel('USD Coin paid into the pool, $\\Delta y$ (thousand)')
    ax.set_ylabel('Profit (thousand USD)')
    legend_outside_bottom(ax, ncol=1, y=-0.22)
    save_fig('ch16_sem_a3_profit')


def fig_sem_a5():
    """A5: impermanent loss IL_w(r) for w = 0.5 and 0.8, with the second-order approximation."""
    r = np.exp(np.linspace(np.log(0.2), np.log(5), 400))
    fig, ax = plt.subplots(figsize=(5.8, 2.8))
    for w, c in [(0.5, MainBlue), (0.8, Orange)]:
        ax.plot(r, 100 * il_weighted(r, w), color=c, lw=1.7, label=f'Exact, w = {w}')
        ax.plot(r, 100 * (np.exp(-w * (1 - w) * np.log(r) ** 2 / 2) - 1), color=c, lw=1.1, ls='--',
                label=f'Second-order approximation, w = {w}')
        for rr in [2.0, 4.0]:
            ax.scatter([rr], [100 * il_weighted(rr, w)], color=c, s=18, zorder=3)
    ax.axhline(0, color=Gray, lw=0.6)
    ax.set_xscale('log')
    ax.set_xticks([0.25, 0.5, 1, 2, 4])
    ax.set_xticklabels(['0.25', '0.5', '1', '2', '4'])
    ax.set_xlabel('Price ratio $r = P_1/P_0$ (log scale)')
    ax.set_ylabel('Impermanent loss (%)')
    legend_outside_bottom(ax, ncol=2, y=-0.22)
    save_fig('ch16_sem_a5_il')


def fig_sem_a7(tar, d0=285.0):
    """A7: deterministic path of |d_t| from 285 bp: phi_out outside the band, phi_in inside; band c."""
    d, path = d0, [d0]
    for _ in range(10):
        d = d * (tar['phi_out'] if d > tar['c'] else tar['phi_in'])
        path.append(d)
    path = np.array(path)
    fig, ax = plt.subplots(figsize=(5.8, 2.7))
    t = np.arange(len(path))
    ax.plot(t, path, color=MainBlue, lw=1.4, marker='o', ms=4, label='Deterministic path of $|d_t|$ ($\\varepsilon_t = 0$), bp')
    ax.axhline(tar['c'], color=IDAred, lw=1.2, ls='--', label=f"Band edge $\\hat c$ = {tar['c']:.2f} bp")
    ax.set_yscale('log')
    ax.set_xlabel('Days after the close of 11 March 2023')
    ax.set_ylabel('Deviation from 1 USD (bp, log scale)')
    legend_outside_bottom(ax, ncol=2, y=-0.22)
    save_fig('ch16_sem_a7_path')


def fig_sem_b2(S):
    """B2: annualised volatility with bootstrap CI and the weekend / weekday ratio before and after the ETFs."""
    rows = [('Bitcoin', S['B1']), ('Ether', S['B2']['ETH']), ('Solana', S['B2']['SOL'])]
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8))
    x = np.arange(3)
    v = np.array([b['vol'] for _, b in rows])
    lo = np.array([b['vol_lo'] for _, b in rows])
    hi = np.array([b['vol_hi'] for _, b in rows])
    axes[0].bar(x, v, 0.55, color=[Orange, Purple, Teal], yerr=[v - lo, hi - v], capsize=3,
                error_kw=dict(ecolor='black', lw=0.8))
    for i, (_, b) in enumerate(rows):
        axes[0].text(i, hi[i] + 3, f"{b['vol']:.0f}%\n{b['ratio']:.1f}x S&P 500", ha='center', fontsize=7, color='black')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([n for n, _ in rows])
    axes[0].set_ylabel('Annualised volatility (%)')
    axes[0].set_ylim(0, 165)
    for j, (per, c, lab) in enumerate([('pre', MainBlue, 'Before 11 Jan 2024'), ('post', Orange, 'From 11 Jan 2024')]):
        m = np.array([b[f'wr_{per}'] for _, b in rows])
        l_ = np.array([b[f'wr_{per}_lo'] for _, b in rows])
        h_ = np.array([b[f'wr_{per}_hi'] for _, b in rows])
        axes[1].bar(x + (j - 0.5) * 0.36, m, 0.34, color=c, yerr=[m - l_, h_ - m], capsize=3,
                    error_kw=dict(ecolor='black', lw=0.8), label=lab)
    axes[1].axhline(1, color=Gray, lw=0.7, ls='--')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([n for n, _ in rows])
    axes[1].set_ylabel('Weekend / weekday variance')
    fig.tight_layout()
    fig_legend_bottom(fig, ncol=2, y=0.0)
    save_fig('ch16_sem_b2_vol')


def fig_sem_b4(S):
    """B4: threshold-AR slopes inside and outside the band, with 95% intervals, for Tether, USD Coin, Dai."""
    rows = [('Tether', S['B4']['USDT']['full']), ('USD Coin', S['B3']['full']), ('Dai', S['B4']['DAI']['full'])]
    fig, ax = plt.subplots(figsize=(5.8, 2.8))
    x = np.arange(3)
    for j, (k, c, lab) in enumerate([('in', MainBlue, 'Inside the band, $\\hat\\phi_{in}$'),
                                     ('out', IDAred, 'Outside the band, $\\hat\\phi_{out}$')]):
        m = np.array([t[f'phi_{k}'] for _, t in rows])
        se = np.array([t[f'se_{k}'] for _, t in rows])
        ax.bar(x + (j - 0.5) * 0.36, m, 0.34, color=c, yerr=1.96 * se, capsize=3,
               error_kw=dict(ecolor='black', lw=0.8), label=lab)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{n}\n$\\hat c$ = {t['c']:.1f} bp, bootstrap p = {t['p_boot']:.3f}" for n, t in rows], fontsize=7.5)
    ax.set_ylabel('AR slope (95% interval)')
    ax.set_ylim(0, 1.05)
    legend_outside_bottom(ax, ncol=2, y=-0.3)
    save_fig('ch16_sem_b4_tar')


def fig_sem_b56(etf, coin, name):
    """B5 / B6: log of the ETF / coin price ratio (in %, relative to the first day) with the estimated trend and
    the trend implied by the fee of 0.25% a year."""
    p = pd.concat([price(etf), price(coin)], axis=1, join='inner').dropna()
    lr = 100 * np.log(p[etf] / p[coin])
    lr = lr - lr.iloc[0]
    t = (lr.index - lr.index[0]).days / 365.25
    b, a = np.polyfit(t, lr.values, 1)
    fig, ax = plt.subplots(figsize=(6.4, 2.7))
    ax.plot(lr.index, lr.values, color=MainBlue, lw=0.6, alpha=0.8, label=f'$\\ell_t$ = ln({etf} / {coin} price), change since day 1 (%)')
    ax.plot(lr.index, lr.rolling(20).mean().values, color=Teal, lw=1.3, label='20-day moving average of $\\ell_t$')
    ax.plot(lr.index, a + b * t, color=IDAred, lw=1.6, label=f'Fitted trend: {b:.3f}% a year')
    ax.plot(lr.index, a - 0.25 * t, color=Forest, lw=1.2, ls='--', label='Slope of the 0.25% sponsor fee')
    ax.set_ylabel('%')
    ax.tick_params(axis='x', labelsize=8)
    legend_outside_bottom(ax, ncol=2, y=-0.2)
    save_fig(name)


def fig_sem_b78(key, b, name):
    """B7 / B8: Wald statistic W(tau) for a break in all coefficients, at each candidate week."""
    w = 100 * joint_returns([key, 'BTC', 'SPX'], None, freq='W').loc['2021-04-16':END]
    X = np.column_stack([np.ones(len(w)), w['BTC'].values, w['SPX'].values])
    sw = I.sup_wald(w[key].values, X, w.index)
    Wser = pd.Series(sw['W'], index=w.index).dropna()
    fig, ax = plt.subplots(figsize=(6.4, 2.7))
    ax.plot(Wser.index, Wser.values, color=MainBlue, lw=1.2, label='$W(\\tau)$, HAC, break in $(a, \\beta_{BTC}, \\beta_{SPX})$')
    ax.axhline(b['sw']['cv5'], color=IDAred, lw=1.1, ls='--', label=f"5% critical value of sup W: {b['sw']['cv5']:.2f}")
    d = pd.Timestamp(b['sw']['date'])
    ax.axvspan(pd.Timestamp(b['sw']['ci_lo']), pd.Timestamp(b['sw']['ci_hi']), color=Amber, alpha=0.2,
               label=f"Break date {d.strftime('%d %b %Y')} and 95% interval")
    ax.axvline(d, color=Amber, lw=1.2)
    ax.axvline(pd.Timestamp(ETF_START), color=Forest, lw=1.1, ls=':', label='Spot ETF launch, 11 Jan 2024')
    ax.set_ylabel('Wald statistic')
    ax.tick_params(axis='x', labelsize=8)
    legend_outside_bottom(ax, ncol=2, y=-0.2)
    save_fig(name)
    return float(np.nanmax(sw['W']))


def fig_sem_c2(c2):
    """C2: Bitcoin's volatility and Sharpe ratio after each correction of the AI answer."""
    labs = ['AI answer\n(252, log returns, $r_f$ = 5%)', 'Annualised with 365\n(same $r_f$)',
            'Corrected\n(365, simple returns, T-bill $r_f$)']
    vol = [c2['sd252'], c2['sd365'], c2['sd_s']]
    sh = [c2['sh252'], c2['sh365'], c2['sh_s']]
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.7))
    x = np.arange(3)
    axes[0].bar(x, vol, 0.55, color=[IDAred, Amber, Forest])
    axes[1].bar(x, sh, 0.55, color=[IDAred, Amber, Forest])
    for i in range(3):
        axes[0].text(i, vol[i] + 1, f'{vol[i]:.1f}%', ha='center', fontsize=7.5, color='black')
        axes[1].text(i, sh[i] + 0.01, f'{sh[i]:.2f}', ha='center', fontsize=7.5, color='black')
    for ax, yl in [(axes[0], 'Annualised volatility (%)'), (axes[1], 'Sharpe ratio')]:
        ax.set_xticks(x)
        ax.set_xticklabels(labs, fontsize=6.5)
        ax.set_ylabel(yl)
    axes[0].set_ylim(0, 70)
    axes[1].set_ylim(0, 0.75)
    fig.tight_layout()
    save_fig('ch16_sem_c2_correction')

## Run

In [ ]:
import types
I = types.SimpleNamespace(tar_fit=tar_fit, nw_lags=nw_lags, sup_wald=sup_wald, sup_wald_crit=sup_wald_crit, dimson=dimson)
for f in [a1_index, a2_replace, a3_band, a4_weighted, a5_il, a6_lvr]:
    print(f.__name__, f())
b1 = b1_btc_facts('BTC')
fig_sem_weekend(b1)
print({k: v for k, v in b1.items() if not k.startswith('_')})
b3 = b3_usdc('USDC')
print('B3', b3)
fig_sem_usdc(b3['full'])
print('A7', a7_tar(b3['full']), 'A8', a8_mix(b3['full']))
print('B5', b5_etf('IBIT', 'BTC'))
print('B7', b7_equity('MSTR'))
c1 = c1_hedge()
fig_sem_hedge(c1)
print(c1['bl'], c1['fr'], c1['qr'])
c2 = c2_ai()
print('C2', c2)
# charts of the solutions
S = dict(A1=a1_index(), A3=a3_band(), B1=b1, B2={k: b1_btc_facts(k) for k in ['ETH', 'SOL']}, B3=b3,
         B4={k: b3_usdc(k) for k in ['USDT', 'DAI']}, B7=b7_equity('MSTR'), B8=b7_equity('COIN'), C2=c2)
fig_sem_a1(S['A1']); fig_sem_a3(S['A3']); fig_sem_a5(); fig_sem_a7(b3['full'])
fig_sem_b2(S); fig_sem_b4(S)
fig_sem_b56('IBIT', 'BTC', 'ch16_sem_b5_ratio'); fig_sem_b56('ETHA', 'ETH', 'ch16_sem_b6_ratio')
fig_sem_b78('MSTR', S['B7'], 'ch16_sem_b7_break'); fig_sem_b78('COIN', S['B8'], 'ch16_sem_b8_break')
fig_sem_c2(c2)

![ch16_sem_a1_index](./ch16_sem_a1_index.png)

Output: `ch16_sem_a1_index.pdf`

![ch16_sem_a3_profit](./ch16_sem_a3_profit.png)

Output: `ch16_sem_a3_profit.pdf`

![ch16_sem_a5_il](./ch16_sem_a5_il.png)

Output: `ch16_sem_a5_il.pdf`

![ch16_sem_a7_path](./ch16_sem_a7_path.png)

Output: `ch16_sem_a7_path.pdf`

![ch16_sem_weekend](./ch16_sem_weekend.png)

Output: `ch16_sem_weekend.pdf`

![ch16_sem_b2_vol](./ch16_sem_b2_vol.png)

Output: `ch16_sem_b2_vol.pdf`

![ch16_sem_usdc](./ch16_sem_usdc.png)

Output: `ch16_sem_usdc.pdf`

![ch16_sem_b4_tar](./ch16_sem_b4_tar.png)

Output: `ch16_sem_b4_tar.pdf`

![ch16_sem_b5_ratio](./ch16_sem_b5_ratio.png)

Output: `ch16_sem_b5_ratio.pdf`

![ch16_sem_b6_ratio](./ch16_sem_b6_ratio.png)

Output: `ch16_sem_b6_ratio.pdf`

![ch16_sem_b7_break](./ch16_sem_b7_break.png)

Output: `ch16_sem_b7_break.pdf`

![ch16_sem_b8_break](./ch16_sem_b8_break.png)

Output: `ch16_sem_b8_break.pdf`

![ch16_sem_hedge](./ch16_sem_hedge.png)

Output: `ch16_sem_hedge.pdf`

![ch16_sem_c2_correction](./ch16_sem_c2_correction.png)

Output: `ch16_sem_c2_correction.pdf`